# AI Coach — POC: Import Garmin Activities via Garmin Connect API

Credentials werden sicher per Eingabe abgefragt. Die Aktivitäten werden direkt von
Garmin Connect heruntergeladen und als `.fit` Dateien in `data/activities/` gespeichert.

In [1]:
%pip install garminconnect fitparse pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import getpass
import io
import zipfile
from pathlib import Path
import fitparse
import pandas as pd
from garminconnect import Garmin

# --- Config ---
ACTIVITIES_DIR = Path("../data/activities")
ACTIVITIES_DIR.mkdir(parents=True, exist_ok=True)
TOKEN_STORE = Path("../data/.garmin_tokens")
NUM_ACTIVITIES = 100

# --- Login (mit Token-Cache) ---
client = Garmin()
try:
    client.login(tokenstore=str(TOKEN_STORE))
    print("Login mit gespeichertem Token erfolgreich.")
except Exception:
    email = input("Garmin Connect E-Mail: ")
    password = getpass.getpass("Passwort: ")
    client = Garmin(email, password)
    client.login()
    client.garth.dump(str(TOKEN_STORE))
    print("Login erfolgreich. Token für künftige Sitzungen gespeichert.")

# --- FIT-Dateien herunterladen und entpacken ---
activities = client.get_activities(0, NUM_ACTIVITIES)

for act in activities:
    act_id = act["activityId"]
    act_name = act.get("activityName", str(act_id))
    fit_path = ACTIVITIES_DIR / f"{act_id}.fit"
    if not fit_path.exists():
        zip_data = client.download_activity(act_id, dl_fmt=client.ActivityDownloadFormat.ORIGINAL)
        with zipfile.ZipFile(io.BytesIO(zip_data)) as zf:
            fit_names = [n for n in zf.namelist() if n.endswith(".fit")]
            if fit_names:
                fit_path.write_bytes(zf.read(fit_names[0]))
                print(f"  Heruntergeladen: {fit_path.name}  ({act_name})")
            else:
                print(f"  Keine .fit in ZIP für Aktivität {act_id} ({act_name})")
    else:
        print(f"  Bereits vorhanden: {fit_path.name}")

fit_files = sorted(ACTIVITIES_DIR.glob("*.fit"))
print(f"\nGesamt: {len(fit_files)} .fit Datei(en) in {ACTIVITIES_DIR.resolve()}")

In [5]:
def parse_fit_file(fit_path: Path) -> pd.DataFrame:
    """Parse a single .fit file and return a DataFrame of 'record' messages."""
    fit = fitparse.FitFile(str(fit_path))
    rows = []
    for msg in fit.get_messages("record"):
        row = {data.name: data.value for data in msg}
        rows.append(row)
    return pd.DataFrame(rows)

In [7]:
fit_files[0]

WindowsPath('../data/activities/22330086513.fit')

In [6]:
# Parse the first available file as a quick test
if fit_files:
    df = parse_fit_file(fit_files[0])
    print(f"Parsed: {fit_files[0].name}  →  {len(df)} records, {len(df.columns)} fields")
    print("\nColumns:", list(df.columns))
    df.head()
else:
    print("No .fit files found. Copy files from your Garmin device to:", ACTIVITIES_DIR.resolve())

FitHeaderError: Invalid .FIT File Header

In [ ]:
# Parse ALL .fit files into a single combined DataFrame
all_dfs = []
for f in fit_files:
    df_tmp = parse_fit_file(f)
    df_tmp.insert(0, "source_file", f.name)
    all_dfs.append(df_tmp)

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f"Total records across all activities: {len(df_all)}")
    df_all.head()
else:
    print("No .fit files to process.")